## Inception Model initialization and test with sample image

In [ ]:
import torch
import torch.nn as nn
import torch.utils.model_zoo as model_zoo
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
import os
from PIL import Image

class BasicConv2d(nn.Module):
    def __init__(self, in_planes, out_planes, kernel_size, stride, padding=0):
        super(BasicConv2d, self).__init__()
        self.conv = nn.Conv2d(in_planes, out_planes,
                              kernel_size=kernel_size, stride=stride,
                              padding=padding, bias=False)
        self.bn = nn.BatchNorm2d(out_planes)
        self.relu = nn.ReLU(inplace=False)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        return x

class Mixed_5b(nn.Module):
    def __init__(self):
        super(Mixed_5b, self).__init__()
        self.branch0 = BasicConv2d(192, 96, kernel_size=1, stride=1)
        self.branch1 = nn.Sequential(
            BasicConv2d(192, 48, kernel_size=1, stride=1),
            BasicConv2d(48, 64, kernel_size=5, stride=1, padding=2)
        )
        self.branch2 = nn.Sequential(
            BasicConv2d(192, 64, kernel_size=1, stride=1),
            BasicConv2d(64, 96, kernel_size=3, stride=1, padding=1),
            BasicConv2d(96, 96, kernel_size=3, stride=1, padding=1)
        )
        self.branch3 = nn.Sequential(
            nn.AvgPool2d(3, stride=1, padding=1, count_include_pad=False),
            BasicConv2d(192, 64, kernel_size=1, stride=1)
        )

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        out = torch.cat((x0, x1, x2, x3), 1)
        return out

class Block35(nn.Module):
    def __init__(self, scale=1.0):
        super(Block35, self).__init__()
        self.scale = scale
        self.branch0 = BasicConv2d(320, 32, kernel_size=1, stride=1)
        self.branch1 = nn.Sequential(
            BasicConv2d(320, 32, kernel_size=1, stride=1),
            BasicConv2d(32, 32, kernel_size=3, stride=1, padding=1)
        )
        self.branch2 = nn.Sequential(
            BasicConv2d(320, 32, kernel_size=1, stride=1),
            BasicConv2d(32, 48, kernel_size=3, stride=1, padding=1),
            BasicConv2d(48, 64, kernel_size=3, stride=1, padding=1)
        )
        self.conv2d = nn.Conv2d(128, 320, kernel_size=1, stride=1)
        self.relu = nn.ReLU(inplace=False)

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        out = torch.cat((x0, x1, x2), 1)
        out = self.conv2d(out)
        out = out * self.scale + x
        out = self.relu(out)
        return out

class Mixed_6a(nn.Module):
    def __init__(self):
        super(Mixed_6a, self).__init__()
        self.branch0 = BasicConv2d(320, 384, kernel_size=3, stride=2)
        self.branch1 = nn.Sequential(
            BasicConv2d(320, 256, kernel_size=1, stride=1),
            BasicConv2d(256, 256, kernel_size=3, stride=1, padding=1),
            BasicConv2d(256, 384, kernel_size=3, stride=2)
        )
        self.branch2 = nn.MaxPool2d(3, stride=2)

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        out = torch.cat((x0, x1, x2), 1)
        return out

class Block17(nn.Module):
    def __init__(self, scale=1.0):
        super(Block17, self).__init__()
        self.scale = scale
        self.branch0 = BasicConv2d(1088, 192, kernel_size=1, stride=1)
        self.branch1 = nn.Sequential(
            BasicConv2d(1088, 128, kernel_size=1, stride=1),
            BasicConv2d(128, 160, kernel_size=(1, 7), stride=1, padding=(0, 3)),
            BasicConv2d(160, 192, kernel_size=(7, 1), stride=1, padding=(3, 0))
        )
        self.conv2d = nn.Conv2d(384, 1088, kernel_size=1, stride=1)
        self.relu = nn.ReLU(inplace=False)

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        out = torch.cat((x0, x1), 1)
        out = self.conv2d(out)
        out = out * self.scale + x
        out = self.relu(out)
        return out

class Mixed_7a(nn.Module):
    def __init__(self):
        super(Mixed_7a, self).__init__()
        self.branch0 = nn.Sequential(
            BasicConv2d(1088, 256, kernel_size=1, stride=1),
            BasicConv2d(256, 384, kernel_size=3, stride=2)
        )
        self.branch1 = nn.Sequential(
            BasicConv2d(1088, 256, kernel_size=1, stride=1),
            BasicConv2d(256, 288, kernel_size=3, stride=2)
        )
        self.branch2 = nn.Sequential(
            BasicConv2d(1088, 256, kernel_size=1, stride=1),
            BasicConv2d(256, 288, kernel_size=3, stride=1, padding=1),
            BasicConv2d(288, 320, kernel_size=3, stride=2)
        )
        self.branch3 = nn.MaxPool2d(3, stride=2)

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        x2 = self.branch2(x)
        x3 = self.branch3(x)
        out = torch.cat((x0, x1, x2, x3), 1)
        return out

class Block8(nn.Module):
    def __init__(self, scale=1.0, noReLU=False):
        super(Block8, self).__init__()
        self.scale = scale
        self.noReLU = noReLU

        self.branch0 = BasicConv2d(2080, 192, kernel_size=1, stride=1)

        self.branch1 = nn.Sequential(
            BasicConv2d(2080, 192, kernel_size=1, stride=1),
            BasicConv2d(192, 224, kernel_size=(1, 3), stride=1, padding=(0, 1)),
            BasicConv2d(224, 256, kernel_size=(3, 1), stride=1, padding=(1, 0))
        )

        self.conv2d = nn.Conv2d(448, 2080, kernel_size=1, stride=1)
        if not self.noReLU:
            self.relu = nn.ReLU(inplace=False)

    def forward(self, x):
        x0 = self.branch0(x)
        x1 = self.branch1(x)
        out = torch.cat((x0, x1), 1)
        out = self.conv2d(out)
        out = out * self.scale + x
        if not self.noReLU:
            out = self.relu(out)
        return out

class InceptionResNetV2(nn.Module):
    def __init__(self, num_classes=1001):
        super(InceptionResNetV2, self).__init__()
        self.conv2d_1a = BasicConv2d(3, 32, kernel_size=3, stride=2)
        self.conv2d_2a = BasicConv2d(32, 32, kernel_size=3, stride=1)
        self.conv2d_2b = BasicConv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.maxpool_3a = nn.MaxPool2d(3, stride=2)
        self.conv2d_3b = BasicConv2d(64, 80, kernel_size=1, stride=1)
        self.conv2d_4a = BasicConv2d(80, 192, kernel_size=3, stride=1)
        self.maxpool_5a = nn.MaxPool2d(3, stride=2)
        self.mixed_5b = Mixed_5b()
        self.repeat = nn.Sequential(
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17)
        )
        self.mixed_6a = Mixed_6a()
        self.repeat_1 = nn.Sequential(
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10)
        )
        self.mixed_7a = Mixed_7a()
        self.repeat_2 = nn.Sequential(
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20)
        )
        self.block8 = Block8(noReLU=True)
        self.conv2d_7b = BasicConv2d(2080, 1536, kernel_size=1, stride=1)
        self.avgpool_1a = nn.AvgPool2d(8, count_include_pad=False)
        self.last_linear = nn.Linear(1536, num_classes)

    def features(self, input):
        x = self.conv2d_1a(input)
        x = self.conv2d_2a(x)
        x = self.conv2d_2b(x)
        x = self.maxpool_3a(x)
        x = self.conv2d_3b(x)
        x = self.conv2d_4a(x)
        x = self.maxpool_5a(x)
        x = self.mixed_5b(x)
        x = self.repeat(x)
        x = self.mixed_6a(x)
        x = self.repeat_1(x)
        x = self.mixed_7a(x)
        x = self.repeat_2(x)
        x = self.block8(x)
        x = self.conv2d_7b(x)
        return x

    def logits(self, features):
        x = self.avgpool_1a(features)
        x = x.view(x.size(0), -1)
        x = self.last_linear(x)
        return x

    def forward(self, input):
        x = self.features(input)
        x = self.logits(x)
        return x

def load_model(model_path, num_classes=4):
    model = InceptionResNetV2(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path))
    model.eval()  # Set the model to evaluation mode
    return model

def preprocess_image(image_path):
    # Define the transformations (same as during training)
    transform = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.CenterCrop(299),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])
    
    # Load the image
    image = Image.open(image_path).convert("RGB")
    image = transform(image)  # Apply transformations
    image = image.unsqueeze(0)  # Add a batch dimension
    return image

def predict(model, image_tensor):
    with torch.no_grad():  # Disable gradient computation
        output = model(image_tensor)
        _, predicted = torch.max(output, 1)  # Get the predicted class
        return predicted.item()  # Return the class index

# Load the model
model_path = r"D:\Samsthidhaa\Semester_7\Project_Phase_1\dataset\eye_gaze_model_2.pth"  # Path to your saved model
model = load_model(model_path)

# Path to the test image
test_image_path = r"D:\Samsthidhaa\Semester_7\Project_Phase_1\left1.png"  # Replace with your actual image path

# Preprocess the image
image_tensor = preprocess_image(test_image_path)

# Make a prediction
predicted_class = predict(model, image_tensor)

# Define the class mapping
class_mapping = {0: 'straight', 1: 'left', 2: 'right', 3: 'closed'}

# Print the predicted class
print(f'Predicted class: {class_mapping[predicted_class]}')


Predicted class: left


## Live model testing

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image
from ultralytics import YOLO
from torchvision import transforms

# Load YOLOv8 model for eye detection
yolo_model = YOLO(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\Eye_Alone_Detect.pt")

class InceptionResNetV2(nn.Module):
    def __init__(self, num_classes=1001):
        super(InceptionResNetV2, self).__init__()
        self.conv2d_1a = BasicConv2d(3, 32, kernel_size=3, stride=2)
        self.conv2d_2a = BasicConv2d(32, 32, kernel_size=3, stride=1)
        self.conv2d_2b = BasicConv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.maxpool_3a = nn.MaxPool2d(3, stride=2)
        self.conv2d_3b = BasicConv2d(64, 80, kernel_size=1, stride=1)
        self.conv2d_4a = BasicConv2d(80, 192, kernel_size=3, stride=1)
        self.maxpool_5a = nn.MaxPool2d(3, stride=2)
        self.mixed_5b = Mixed_5b()
        self.repeat = nn.Sequential(
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17)
        )
        self.mixed_6a = Mixed_6a()
        self.repeat_1 = nn.Sequential(
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10)
        )
        self.mixed_7a = Mixed_7a()
        self.repeat_2 = nn.Sequential(
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20)
        )
        self.block8 = Block8(noReLU=True)
        self.conv2d_7b = BasicConv2d(2080, 1536, kernel_size=1, stride=1)
        self.avgpool_1a = nn.AvgPool2d(8, count_include_pad=False)
        self.last_linear = nn.Linear(1536, num_classes)

    def features(self, input):
        x = self.conv2d_1a(input)
        x = self.conv2d_2a(x)
        x = self.conv2d_2b(x)
        x = self.maxpool_3a(x)
        x = self.conv2d_3b(x)
        x = self.conv2d_4a(x)
        x = self.maxpool_5a(x)
        x = self.mixed_5b(x)
        x = self.repeat(x)
        x = self.mixed_6a(x)
        x = self.repeat_1(x)
        x = self.mixed_7a(x)
        x = self.repeat_2(x)
        x = self.block8(x)
        x = self.conv2d_7b(x)
        return x

    def logits(self, features):
        x = self.avgpool_1a(features)
        x = x.view(x.size(0), -1)
        x = self.last_linear(x)
        return x

    def forward(self, input):
        x = self.features(input)
        x = self.logits(x)
        return x

def load_model(model_path, num_classes=4):
    model = InceptionResNetV2(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model

gaze_model = load_model(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\dataset\eye_gaze_model_2.pth")

# Preprocess the ROI (detected eye)
def preprocess_image(roi):
    transform = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])
    image = Image.fromarray(roi)
    image = transform(image).unsqueeze(0)  # Add batch dimension
    return image

# Predict gaze using InceptionResNetV2
def predict_gaze(model, image_tensor):
    with torch.no_grad():
        output = model(image_tensor)
        _, predicted = torch.max(output, 1)
        return predicted.item()

# Class mapping for gaze direction
class_mapping = {0: 'straight', 1: 'left', 2: 'right', 3: 'closed'}

cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    results = yolo_model(frame)
    
    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            roi = frame[y1:y2, x1:x2]
            
            roi_tensor = preprocess_image(roi)
            
            gaze_class = predict_gaze(gaze_model, roi_tensor)
            gaze_label = class_mapping[gaze_class]
            
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, gaze_label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36, 255, 12), 2)

    cv2.imshow('Eye Gaze Detection', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()



0: 480x640 (no detections), 78.2ms
Speed: 2.0ms preprocess, 78.2ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 45.3ms
Speed: 2.5ms preprocess, 45.3ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 42.6ms
Speed: 3.0ms preprocess, 42.6ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 43.0ms
Speed: 3.0ms preprocess, 43.0ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 42.5ms
Speed: 2.0ms preprocess, 42.5ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 42.2ms
Speed: 2.5ms preprocess, 42.2ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 41.4ms
Speed: 2.5ms preprocess, 41.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 41.8ms
Speed: 2.0ms preprocess, 41.8ms i

Speed: 2.0ms preprocess, 137.7ms inference, 5.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 46.0ms
Speed: 4.0ms preprocess, 46.0ms inference, 5.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 46.8ms
Speed: 1.5ms preprocess, 46.8ms inference, 4.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 146.0ms
Speed: 2.4ms preprocess, 146.0ms inference, 4.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 46.6ms
Speed: 2.0ms preprocess, 46.6ms inference, 4.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 47.2ms
Speed: 2.0ms preprocess, 47.2ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 135.4ms
Speed: 4.0ms preprocess, 135.4ms inference, 4.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 45.6ms
Speed: 2.5ms preprocess, 45.6ms inference, 4.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 46.1ms
Speed: 3.0m


0: 480x640 1 eye, 45.6ms
Speed: 3.0ms preprocess, 45.6ms inference, 4.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 46.6ms
Speed: 2.0ms preprocess, 46.6ms inference, 4.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 131.7ms
Speed: 3.5ms preprocess, 131.7ms inference, 6.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 51.5ms
Speed: 3.0ms preprocess, 51.5ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 51.3ms
Speed: 2.5ms preprocess, 51.3ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 133.4ms
Speed: 3.0ms preprocess, 133.4ms inference, 4.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 55.8ms
Speed: 4.5ms preprocess, 55.8ms inference, 3.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 eye, 55.5ms
Speed: 4.0ms preprocess, 55.5ms inference, 5.0ms postprocess per image at shape (1, 3, 480, 640)


## Definition of Communication System - Blink detection

In [ ]:
def comm():

    import pyttsx3
    import cv2
    from ultralytics import YOLO
    import time

    tts_engine = pyttsx3.init()

    def play_tts_message(message):
        tts_engine.say(message)
        tts_engine.runAndWait()

    model = YOLO(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\bestEyeyolov8.pt")

    cap = cv2.VideoCapture(0)  
    assert cap.isOpened(), "Error reading video file"

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    video_writer = cv2.VideoWriter("blink_count_output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    blink_count = 0
    eye_was_closed = False
    blink_start_time = 0  # Time when the eye was detected as closed
    closed_time = 0  # Time counter for how long eyes have been closed
    blinks_detected = 0  # Counter for detected blinks
    selected_water = False  # Flag for water selection
    display_selection_text = False  # Flag for displaying the text
    confirm_phase = False  # Flag for the confirmation phase

    def draw_eye_bounding_box_and_line(frame, bbox):
        x1, y1, x2, y2 = map(int, bbox) 
        center_y = int((y1 + y2) / 2) 

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  # Red bounding box

        line_start = (x1, center_y)
        line_end = (x2, center_y)
        cv2.line(frame, line_start, line_end, (0, 255, 0), 2)  # Green line

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("Video frame is empty or video processing is done.")
            break

        results = model(frame)

        eye_detected = False

        for r in results[0].boxes:
            bbox = r.xyxy[0].tolist() 
            class_id = int(r.cls[0])
            eye_label = results[0].names[class_id]  # 'Closed Eyes' or 'Opened Eyes'

            draw_eye_bounding_box_and_line(frame, bbox)

            if eye_label == "closed eyes":
                eye_detected = True
                if not eye_was_closed:
                    blink_start_time = cv2.getTickCount()  # Record the time when eyes were detected as closed
                    eye_was_closed = True 
                    closed_time = 0 
                else:
                    closed_time += 1 / fps  # Increment based on frame rate

            elif eye_label == "opened eyes":
                eye_detected = True
                if eye_was_closed:
                    blink_duration = (cv2.getTickCount() - blink_start_time) / cv2.getTickFrequency()
                    if blink_duration > 0.2:  
                        blink_count += 1 
                        blinks_detected += 1  
                        print(f"Blinked! Count: {blink_count}")
                    eye_was_closed = False 

        if not eye_detected:
            eye_was_closed = False

        # Check if 4 blinks are detected
        if blinks_detected == 4 and not selected_water:
            display_selection_text = True 
            cv2.putText(frame, "You selected water!", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, "Blink twice to confirm", (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            selected_water = True  # Mark as selected water
            confirm_phase = True  # Enter confirmation phase
            blinks_detected = 0  # Reset the blink counter for confirmation

        # Check for 2 more blinks to confirm
        if confirm_phase and blinks_detected == 2:
            display_selection_text = True  
            cv2.putText(frame, "You selected water!", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

            break  

        if display_selection_text:
            cv2.putText(frame, "You selected water!", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            if confirm_phase:
                cv2.putText(frame, "Blink twice to confirm", (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        cv2.putText(frame, f"Blinks: {blink_count}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        cv2.imshow("Blink Counter", frame)

        video_writer.write(frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    video_writer.release()
    cv2.destroyAllWindows()

    play_tts_message("I need water, please provide.")

    print(f"Total blinks counted: {blink_count}")


## Communication system with live video testing

In [ ]:
def comm(video_source=None):
    import pyttsx3
    import cv2
    from ultralytics import YOLO
    import time

    tts_engine = pyttsx3.init()

    def play_tts_message(message):
        tts_engine.say(message)
        tts_engine.runAndWait()

    model = YOLO(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\bestEyeyolov8.pt")

    if video_source:
        cap = cv2.VideoCapture(video_source)  
    else:
        cap = cv2.VideoCapture(0) 

    assert cap.isOpened(), "Error opening video source"

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    video_writer = cv2.VideoWriter("blink_count_output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    blink_count = 0
    eye_was_closed = False
    blink_start_time = 0  # Time when the eye was detected as closed
    closed_time = 0  # Time counter for how long eyes have been closed
    blinks_detected = 0  # Counter for detected blinks
    selected_water = False  # Flag for water selection
    display_selection_text = False  # Flag for displaying the text
    confirm_phase = False  # Flag for the confirmation phase

    def draw_eye_bounding_box_and_line(frame, bbox):
        x1, y1, x2, y2 = map(int, bbox)  
        center_y = int((y1 + y2) / 2) 

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  # Red bounding box

        line_start = (x1, center_y)
        line_end = (x2, center_y)
        cv2.line(frame, line_start, line_end, (0, 255, 0), 2)  # Green line

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("Video frame is empty or video processing is done.")
            break

        results = model(frame)

        eye_detected = False

        for r in results[0].boxes:
            bbox = r.xyxy[0].tolist()  
            class_id = int(r.cls[0])
            eye_label = results[0].names[class_id]  # 'Closed Eyes' or 'Opened Eyes'

            draw_eye_bounding_box_and_line(frame, bbox)

            if eye_label == "closed eyes":
                eye_detected = True
                if not eye_was_closed:
                    blink_start_time = cv2.getTickCount()  # Record the time when eyes were detected as closed
                    eye_was_closed = True  
                    closed_time = 0  
                else:
                    closed_time += 1 / fps  

            elif eye_label == "opened eyes":
                eye_detected = True
                if eye_was_closed:
                    blink_duration = (cv2.getTickCount() - blink_start_time) / cv2.getTickFrequency()
                    if blink_duration > 0.2:  # 0.2 seconds threshold
                        blink_count += 1  # Increment blink count
                        blinks_detected += 1  
                        print(f"Blinked! Count: {blink_count}")
                    eye_was_closed = False 

        if not eye_detected:
            eye_was_closed = False

        # Check if 4 blinks are detected
        if blinks_detected == 4 and not selected_water:
            display_selection_text = True  
            cv2.putText(frame, "You selected water!", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            cv2.putText(frame, "Blink twice to confirm", (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            selected_water = True 
            confirm_phase = True 
            blinks_detected = 0 

        # Check for 2 more blinks to confirm
        if confirm_phase and blinks_detected == 2:
            display_selection_text = True  
            cv2.putText(frame, "You selected water!", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

            break  

        # If selection or confirmation phase is active, display the text
        if display_selection_text:
            cv2.putText(frame, "You selected water!", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            if confirm_phase:
                cv2.putText(frame, "Blink twice to confirm", (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        cv2.putText(frame, f"Blinks: {blink_count}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        cv2.imshow("Blink Counter", frame)

        video_writer.write(frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    video_writer.release()
    cv2.destroyAllWindows()

    play_tts_message("I need water, please provide.")

    print(f"Total blinks counted: {blink_count}")

# To use webcam (default):
# comm()

# To use a video file (provide the path):
# comm("path_to_your_video.mp4")


## Final Project Code : Blink and Gaze detection with respective controls

In [13]:
import cv2
from ultralytics import YOLO
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
import torch.nn as nn

model = YOLO(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\bestEyeyolov8.pt")

yolo_model = YOLO(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\Eye_Alone_Detect.pt")

cap = cv2.VideoCapture(0)  
assert cap.isOpened(), "Error reading video file"

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

video_writer = cv2.VideoWriter("blink_count_output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

blink_count = 0
eye_was_closed = False
closed_time = 0  
blinks_detected = 0  

class InceptionResNetV2(nn.Module):
    def __init__(self, num_classes=1001):
        super(InceptionResNetV2, self).__init__()
        self.conv2d_1a = BasicConv2d(3, 32, kernel_size=3, stride=2)
        self.conv2d_2a = BasicConv2d(32, 32, kernel_size=3, stride=1)
        self.conv2d_2b = BasicConv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.maxpool_3a = nn.MaxPool2d(3, stride=2)
        self.conv2d_3b = BasicConv2d(64, 80, kernel_size=1, stride=1)
        self.conv2d_4a = BasicConv2d(80, 192, kernel_size=3, stride=1)
        self.maxpool_5a = nn.MaxPool2d(3, stride=2)
        self.mixed_5b = Mixed_5b()
        self.repeat = nn.Sequential(
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17)
        )
        self.mixed_6a = Mixed_6a()
        self.repeat_1 = nn.Sequential(
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10)
        )
        self.mixed_7a = Mixed_7a()
        self.repeat_2 = nn.Sequential(
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20)
        )
        self.block8 = Block8(noReLU=True)
        self.conv2d_7b = BasicConv2d(2080, 1536, kernel_size=1, stride=1)
        self.avgpool_1a = nn.AvgPool2d(8, count_include_pad=False)
        self.last_linear = nn.Linear(1536, num_classes)

    def features(self, input):
        x = self.conv2d_1a(input)
        x = self.conv2d_2a(x)
        x = self.conv2d_2b(x)
        x = self.maxpool_3a(x)
        x = self.conv2d_3b(x)
        x = self.conv2d_4a(x)
        x = self.maxpool_5a(x)
        x = self.mixed_5b(x)
        x = self.repeat(x)
        x = self.mixed_6a(x)
        x = self.repeat_1(x)
        x = self.mixed_7a(x)
        x = self.repeat_2(x)
        x = self.block8(x)
        x = self.conv2d_7b(x)
        return x

    def logits(self, features):
        x = self.avgpool_1a(features)
        x = x.view(x.size(0), -1)
        x = self.last_linear(x)
        return x

    def forward(self, input):
        x = self.features(input)
        x = self.logits(x)
        return x

def load_model(model_path, num_classes=4):
    model = InceptionResNetV2(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model

gaze_model = load_model(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\dataset\eye_gaze_model_2.pth")

def preprocess_image(roi):
    transform = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])
    image = Image.fromarray(roi)
    image = transform(image).unsqueeze(0)  # Add batch dimension
    return image

def predict_gaze(model, image_tensor):
    with torch.no_grad():
        output = model(image_tensor)
        _, predicted = torch.max(output, 1)
        return predicted.item()

class_mapping = {0: 'straight', 1: 'left', 2: 'right', 3: 'closed'}

def draw_eye_bounding_box_and_line(frame, bbox):
    x1, y1, x2, y2 = map(int, bbox) 
    center_y = int((y1 + y2) / 2)  

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  
    
    line_start = (x1, center_y)
    line_end = (x2, center_y)
    cv2.line(frame, line_start, line_end, (0, 255, 0), 2) 
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("Video frame is empty or video processing is done.")
        break
    
    results = model(frame)
    
    eye_detected = False

    for r in results[0].boxes:
        bbox = r.xyxy[0].tolist()  
        class_id = int(r.cls[0])
        eye_label = results[0].names[class_id]  
        
        print(f"Detected label: {eye_label}, BBox: {bbox}") 
        
        draw_eye_bounding_box_and_line(frame, bbox)
        
        if eye_label == "closed eyes":
            eye_detected = True
            if not eye_was_closed:
                eye_was_closed = True  
                closed_time = 0  
            else:
                closed_time += 1 / fps 
            
            if closed_time >= 4:  
                print("Entered blink detection")
                cap.release() 
                video_writer.release() 
                cv2.destroyAllWindows() 
                comm()
                
        
        elif eye_label == "opened eyes":
            eye_detected = True
            if eye_was_closed:
                blink_count += 1  
                blinks_detected += 1  
                print(f"Blinked! Count: {blink_count}") 
                eye_was_closed = False 

    if not eye_detected:
        eye_was_closed = False

    if blinks_detected >= 4:
        print("Mobility mode!")
            
        text_1 = "The Mobility Mode is Activated!!"
        text_2 = "Look Left to Move Left"
        text_3 = "Look Right to Move Right"
        text_4 = "Look Straight to Move Straight"
        text_5 = "Close Eyes to Take Reverse"

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            cv2.putText(frame, text_1, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, text_2, (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, text_3, (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, text_4, (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, text_5, (50, 250), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

            results = yolo_model(frame)
            
            for result in results:
                for box in result.boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                    roi = frame[y1:y2, x1:x2]
                    
                    roi_tensor = preprocess_image(roi)
                    
                    gaze_class = predict_gaze(gaze_model, roi_tensor)
                    gaze_label = class_mapping[gaze_class]
                    
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, gaze_label, (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36, 255, 12), 2)

            cv2.imshow('Eye Gaze Detection', frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        video_writer.release()
        cv2.destroyAllWindows()
        break

    cv2.putText(frame, f"Blinks: {blink_count}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
    
    cv2.imshow("Blink Counter", frame)
    
    video_writer.write(frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()

print(f"Total blinks counted: {blink_count}")



0: 480x640 2 opened eyess, 46.4ms
Speed: 3.0ms preprocess, 46.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)
Detected label: opened eyes, BBox: [239.81793212890625, 252.78915405273438, 303.5614013671875, 286.4298400878906]
Detected label: opened eyes, BBox: [342.94952392578125, 239.58534240722656, 411.10443115234375, 278.451904296875]

0: 480x640 2 opened eyess, 40.9ms
Speed: 3.0ms preprocess, 40.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)
Detected label: opened eyes, BBox: [239.2352294921875, 254.03860473632812, 302.473876953125, 287.8146667480469]
Detected label: opened eyes, BBox: [342.720947265625, 240.02520751953125, 411.27978515625, 279.72528076171875]

0: 480x640 2 opened eyess, 40.9ms
Speed: 3.0ms preprocess, 40.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)
Detected label: opened eyes, BBox: [239.5924072265625, 254.37132263183594, 302.4256591796875, 287.7254638671875]
Detected label: opened eyes, BBox: [342.

0: 480x640 2 closed eyess, 37.9ms
Speed: 3.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)
Detected label: closed eyes, BBox: [243.10165405273438, 246.96722412109375, 303.2756042480469, 289.42852783203125]
Detected label: closed eyes, BBox: [342.7536926269531, 232.98129272460938, 406.4288024902344, 278.4689636230469]

0: 480x640 2 closed eyess, 39.1ms
Speed: 2.0ms preprocess, 39.1ms inference, 4.0ms postprocess per image at shape (1, 3, 480, 640)
Detected label: closed eyes, BBox: [242.79071044921875, 246.83062744140625, 302.8966064453125, 287.77197265625]
Detected label: closed eyes, BBox: [342.9267272949219, 231.68325805664062, 406.6622009277344, 277.4472961425781]

0: 480x640 2 closed eyess, 37.9ms
Speed: 3.0ms preprocess, 37.9ms inference, 4.2ms postprocess per image at shape (1, 3, 480, 640)
Detected label: closed eyes, BBox: [342.74981689453125, 230.18515014648438, 405.96429443359375, 273.7430725097656]
Detected label: closed eyes, BBox: [2

Detected label: closed eyes, BBox: [242.24618530273438, 236.7542266845703, 303.1477355957031, 279.34808349609375]
Detected label: closed eyes, BBox: [347.3301086425781, 224.44537353515625, 414.3291931152344, 267.69036865234375]

0: 480x640 2 closed eyess, 40.4ms
Speed: 2.0ms preprocess, 40.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)
Detected label: closed eyes, BBox: [242.6946563720703, 237.2576904296875, 303.17645263671875, 279.40802001953125]
Detected label: closed eyes, BBox: [347.0048828125, 224.5628662109375, 414.1192626953125, 268.60107421875]

0: 480x640 2 closed eyess, 39.9ms
Speed: 2.0ms preprocess, 39.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)
Detected label: closed eyes, BBox: [347.6395568847656, 224.29791259765625, 413.9430847167969, 267.69073486328125]
Detected label: closed eyes, BBox: [242.98916625976562, 236.98800659179688, 302.3903503417969, 279.1805114746094]

0: 480x640 2 closed eyess, 39.4ms
Speed: 2.0ms preprocess,

Detected label: closed eyes, BBox: [240.82577514648438, 237.5323486328125, 302.7651672363281, 279.86102294921875]
Detected label: closed eyes, BBox: [347.3194580078125, 224.97189331054688, 411.1756591796875, 266.0840148925781]

0: 480x640 2 closed eyess, 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)
Detected label: closed eyes, BBox: [242.2694091796875, 237.94091796875, 302.57098388671875, 279.27587890625]
Detected label: closed eyes, BBox: [345.2578125, 223.29153442382812, 411.26727294921875, 266.8133544921875]

0: 480x640 2 closed eyess, 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)
Detected label: closed eyes, BBox: [242.843017578125, 237.78952026367188, 302.68280029296875, 279.0498962402344]
Detected label: closed eyes, BBox: [345.35333251953125, 222.89581298828125, 411.47552490234375, 266.18212890625]

0: 480x640 2 closed eyess, 39.2ms
Speed: 2.0ms preprocess, 39.2ms


0: 480x640 2 opened eyess, 37.9ms
Speed: 3.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.9ms
Speed: 3.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 39.4ms
Speed: 2.0ms preprocess, 39.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 39.5ms
Speed: 2.0ms preprocess, 39.5ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.9ms
Speed: 3.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 opened eyess, 38.2ms
Speed: 2.0ms preprocess, 38.2ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.9ms
Speed: 3.0ms preprocess, 38.9ms inference

...


0: 480x640 2 closed eyess, 3 opened eyess, 37.9ms
Speed: 2.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 closed eyess, 2 opened eyess, 38.4ms
Speed: 3.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 opened eyess, 38.4ms
Speed: 2.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 opened eyess, 38.9ms
Speed: 3.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 opened eyess, 37.9ms
Speed: 2.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 opened eyess, 39.3ms
Speed: 2.0ms preprocess, 39.3ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 opened eyess, 39.0ms
Speed: 2.0ms preprocess, 39.0ms inference, 3.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 opened eyess, 38.9ms
Speed: 2.

Speed: 2.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 37.9ms
Speed: 2.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)
Blinked! Count: 4

0: 480x640 2 opened eyess, 39.9ms
Speed: 2.0ms preprocess, 39.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.4ms
Speed: 2.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 39.4ms
Speed: 2.0ms preprocess, 39.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.9ms
Speed: 3.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 39.4ms
Speed: 2.0ms preprocess, 39.4ms inference, 3.0ms postproce

Speed: 2.0ms preprocess, 39.4ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 opened eyes, 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 3.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 38.9ms
Speed: 3.0ms preprocess, 38.9ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 38.4ms
Speed: 3.0ms preprocess, 38.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 37.9ms
Speed: 2.0ms preprocess, 37.9ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 37.9ms
Speed: 3.0ms preprocess, 37.9ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 38.4ms
Speed: 2.0ms preprocess, 38.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 2.0ms postprocess per image 

Speed: 2.0ms preprocess, 38.9ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 37.9ms
Speed: 3.0ms preprocess, 37.9ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 38.1ms
Speed: 2.0ms preprocess, 38.1ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 38.4ms
Speed: 2.0ms preprocess, 38.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 38.3ms
Speed: 2.0ms preprocess, 38.3ms inference, 2.0ms postprocess per imag

Speed: 2.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 37.9ms
Speed: 2.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.1ms
Speed: 2.0ms preprocess, 38.1ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.8ms
Speed: 2.0ms preprocess, 38.8ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.9ms
Speed: 3.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.9ms
Speed: 3.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.4ms
Speed: 3.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.4ms
Speed: 3.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at sh

## Final Project Code : Blink and Gaze detection with respective controls - Video Testing

In [ ]:
import cv2
from ultralytics import YOLO
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
import torch.nn as nn

model = YOLO(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\bestEyeyolov8.pt")
yolo_model = YOLO(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\Eye_Alone_Detect.pt")

input_video_path = r"D:\Samsthidhaa\Semester_7\Project_Phase_1\blink test.mp4"
cap = cv2.VideoCapture(input_video_path)  
assert cap.isOpened(), "Error opening video file"

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

output_video_path = r"D:\Samsthidhaa\Semester_7\Project_Phase_1\blinkltestvideooutput.mp4"
video_writer = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

blink_count = 0
eye_was_closed = False
closed_time = 0  
blinks_detected = 0  

class InceptionResNetV2(nn.Module):
    def __init__(self, num_classes=1001):
        super(InceptionResNetV2, self).__init__()
        self.conv2d_1a = BasicConv2d(3, 32, kernel_size=3, stride=2)
        self.conv2d_2a = BasicConv2d(32, 32, kernel_size=3, stride=1)
        self.conv2d_2b = BasicConv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.maxpool_3a = nn.MaxPool2d(3, stride=2)
        self.conv2d_3b = BasicConv2d(64, 80, kernel_size=1, stride=1)
        self.conv2d_4a = BasicConv2d(80, 192, kernel_size=3, stride=1)
        self.maxpool_5a = nn.MaxPool2d(3, stride=2)
        self.mixed_5b = Mixed_5b()
        self.repeat = nn.Sequential(
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17)
        )
        self.mixed_6a = Mixed_6a()
        self.repeat_1 = nn.Sequential(
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10)
        )
        self.mixed_7a = Mixed_7a()
        self.repeat_2 = nn.Sequential(
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20)
        )
        self.block8 = Block8(noReLU=True)
        self.conv2d_7b = BasicConv2d(2080, 1536, kernel_size=1, stride=1)
        self.avgpool_1a = nn.AvgPool2d(8, count_include_pad=False)
        self.last_linear = nn.Linear(1536, num_classes)

    def features(self, input):
        x = self.conv2d_1a(input)
        x = self.conv2d_2a(x)
        x = self.conv2d_2b(x)
        x = self.maxpool_3a(x)
        x = self.conv2d_3b(x)
        x = self.conv2d_4a(x)
        x = self.maxpool_5a(x)
        x = self.mixed_5b(x)
        x = self.repeat(x)
        x = self.mixed_6a(x)
        x = self.repeat_1(x)
        x = self.mixed_7a(x)
        x = self.repeat_2(x)
        x = self.block8(x)
        x = self.conv2d_7b(x)
        return x

    def logits(self, features):
        x = self.avgpool_1a(features)
        x = x.view(x.size(0), -1)
        x = self.last_linear(x)
        return x

    def forward(self, input):
        x = self.features(input)
        x = self.logits(x)
        return x


def load_model(model_path, num_classes=4):
    model = InceptionResNetV2(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model

gaze_model = load_model(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\dataset\eye_gaze_model_2.pth")

def preprocess_image(roi):
    transform = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])
    image = Image.fromarray(roi)
    image = transform(image).unsqueeze(0) 
    return image

def predict_gaze(model, image_tensor):
    with torch.no_grad():
        output = model(image_tensor)
        _, predicted = torch.max(output, 1)
        return predicted.item()

class_mapping = {0: 'straight', 1: 'left', 2: 'right', 3: 'closed'}

def draw_eye_bounding_box_and_line(frame, bbox):
    x1, y1, x2, y2 = map(int, bbox) 
    center_y = int((y1 + y2) / 2)  
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  
    line_start = (x1, center_y)
    line_end = (x2, center_y)
    cv2.line(frame, line_start, line_end, (0, 255, 0), 2)  # Green line

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("End of video or video reading error.")
        break
    
    results = model(frame)
    eye_detected = False

    for r in results[0].boxes:
        bbox = r.xyxy[0].tolist()  
        class_id = int(r.cls[0])
        eye_label = results[0].names[class_id]  
        
        print(f"Detected label: {eye_label}, BBox: {bbox}") 
        
        draw_eye_bounding_box_and_line(frame, bbox)
        
        if eye_label == "closed eyes":
            eye_detected = True
            if not eye_was_closed:
                eye_was_closed = True  
                closed_time = 0  
            else:
                closed_time += 1 / fps 
            
            if closed_time >= 4:  # Check if eyes have been closed for 4 seconds
                print("Entered blink detection")
                cap.release() 
                video_writer.release() 
                cv2.destroyAllWindows()  
                #exit() 
                comm(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\4-2blinks.mp4")
        
        elif eye_label == "opened eyes":
            eye_detected = True
            if eye_was_closed:
                blink_count += 1  
                blinks_detected += 1 
                print(f"Blinked! Count: {blink_count}")  
                eye_was_closed = False 

    if not eye_detected:
        eye_was_closed = False

    # Mobility mode after 4 blinks
    if blinks_detected >= 4:
        print("Mobility mode activated!")
        text_1 = "The Mobility Mode is Activated!!"
        text_2 = "Look Left to Move Left"
        text_3 = "Look Right to Move Right"
        text_4 = "Look Straight to Move Straight"
        text_5 = "Close Eyes to Take Reverse"

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            cv2.putText(frame, text_1, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, text_2, (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, text_3, (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, text_4, (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, text_5, (50, 250), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

            results = yolo_model(frame)
            for result in results:
                for box in result.boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                    roi = frame[y1:y2, x1:x2]
                    roi_tensor = preprocess_image(roi)
                    gaze_class = predict_gaze(gaze_model, roi_tensor)
                    gaze_label = class_mapping[gaze_class]

                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(frame, gaze_label, (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36, 255, 12), 2)

            cv2.imshow('Eye Gaze Detection', frame)
            video_writer.write(frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        cap.release()
        video_writer.release()
        cv2.destroyAllWindows()
        break

    cv2.putText(frame, f"Blinks: {blink_count}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
    cv2.imshow("Blink Counter", frame)
    video_writer.write(frame)  
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()

print(f"Total blinks counted: {blink_count}")



0: 384x640 2 closed eyess, 38.4ms
Speed: 4.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)
Detected label: closed eyes, BBox: [1024.013671875, 429.7079772949219, 1235.146728515625, 549.6655883789062]
Detected label: closed eyes, BBox: [672.8724365234375, 428.9859619140625, 895.9176635742188, 560.409423828125]

0: 384x640 2 closed eyess, 34.9ms
Speed: 4.0ms preprocess, 34.9ms inference, 4.0ms postprocess per image at shape (1, 3, 384, 640)
Detected label: closed eyes, BBox: [1024.107421875, 429.7745361328125, 1235.0321044921875, 549.63134765625]
Detected label: closed eyes, BBox: [672.8935546875, 428.9769287109375, 895.9969482421875, 560.4981079101562]

0: 384x640 2 closed eyess, 32.9ms
Speed: 5.0ms preprocess, 32.9ms inference, 5.0ms postprocess per image at shape (1, 3, 384, 640)
Detected label: closed eyes, BBox: [1024.113037109375, 429.833251953125, 1235.045166015625, 549.6519775390625]
Detected label: closed eyes, BBox: [672.8892822265625, 4

Detected label: closed eyes, BBox: [1024.014404296875, 429.55133056640625, 1235.185546875, 549.76611328125]
Detected label: closed eyes, BBox: [672.9044189453125, 428.9901123046875, 895.9560546875, 560.46826171875]

0: 384x640 2 closed eyess, 30.4ms
Speed: 3.0ms preprocess, 30.4ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)
Detected label: closed eyes, BBox: [1024.0262451171875, 429.69635009765625, 1235.270751953125, 549.7241821289062]
Detected label: closed eyes, BBox: [672.93212890625, 428.93267822265625, 896.0618896484375, 560.4645385742188]

0: 384x640 2 closed eyess, 29.6ms
Speed: 3.0ms preprocess, 29.6ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)
Detected label: closed eyes, BBox: [1024.015869140625, 429.55377197265625, 1235.1837158203125, 549.7662353515625]
Detected label: closed eyes, BBox: [672.9046630859375, 428.99871826171875, 895.9583129882812, 560.4666137695312]

0: 384x640 2 closed eyess, 29.9ms
Speed: 3.0ms preprocess, 29.9ms in

0: 384x640 2 closed eyess, 29.9ms
Speed: 3.0ms preprocess, 29.9ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)
Detected label: closed eyes, BBox: [1024.0362548828125, 429.466796875, 1235.3538818359375, 549.732177734375]
Detected label: closed eyes, BBox: [672.81005859375, 428.8399658203125, 895.9121704101562, 560.4318237304688]

0: 384x640 2 closed eyess, 30.9ms
Speed: 3.0ms preprocess, 30.9ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)
Detected label: closed eyes, BBox: [1024.031494140625, 429.467529296875, 1235.352783203125, 549.7113037109375]
Detected label: closed eyes, BBox: [672.8116455078125, 428.84033203125, 895.913330078125, 560.45361328125]

0: 384x640 2 closed eyess, 29.4ms
Speed: 3.0ms preprocess, 29.4ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)
Detected label: closed eyes, BBox: [1023.9592895507812, 429.4990539550781, 1235.347900390625, 549.696044921875]
Detected label: closed eyes, BBox: [672.815185546875, 42


0: 640x512 2 opened eyess, 37.4ms
Speed: 6.0ms preprocess, 37.4ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 2 opened eyess, 37.9ms
Speed: 5.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 36.9ms
Speed: 6.0ms preprocess, 36.9ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 38.4ms
Speed: 5.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 1 closed eyes, 3 opened eyess, 37.4ms
Speed: 6.0ms preprocess, 37.4ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 closed eyess, 37.9ms
Speed: 5.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 2 closed eyess, 36.9ms
Speed: 7.0ms preprocess, 36.9ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 closed eyess, 37.4ms
Speed: 6.0ms preprocess, 3

Speed: 4.5ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 37.4ms
Speed: 6.0ms preprocess, 37.4ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 37.9ms
Speed: 5.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 37.9ms
Speed: 6.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 37.4ms
Speed: 5.0ms preprocess, 37.4ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 38.4ms
Speed: 6.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 37.9ms
Speed: 6.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 37.9ms
Speed: 6.5ms preprocess, 37.9ms inference, 3.0ms postprocess per image at sh


0: 640x512 3 opened eyess, 38.4ms
Speed: 6.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 37.9ms
Speed: 6.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 36.9ms
Speed: 6.0ms preprocess, 36.9ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 38.4ms
Speed: 6.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 36.9ms
Speed: 7.0ms preprocess, 36.9ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 37.9ms
Speed: 7.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 38.2ms
Speed: 6.5ms preprocess, 38.2ms inference, 3.0ms postprocess per image at shape (1, 3, 640, 512)

0: 640x512 3 opened eyess, 38.1ms
Speed: 6.0ms preprocess, 38.1ms inference

## Final Project Code : Blink and Gaze detection with respective controls - IMAGE INFERENCE

In [ ]:
import cv2
from ultralytics import YOLO
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
import torch.nn as nn

model = YOLO(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\bestEyeyolov8.pt")
yolo_model = YOLO(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\Eye_Alone_Detect.pt")
gaze_model = load_model(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\dataset\eye_gaze_model_2.pth")

class_mapping = {0: 'straight', 1: 'left', 2: 'right', 3: 'closed'}

def preprocess_image(roi):
    transform = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])
    image = Image.fromarray(roi)
    image = transform(image).unsqueeze(0)  
    return image

def predict_gaze(model, image_tensor):
    with torch.no_grad():
        output = model(image_tensor)
        _, predicted = torch.max(output, 1)
        return predicted.item()

def draw_eye_bounding_box_and_line(frame, bbox):
    x1, y1, x2, y2 = map(int, bbox)
    center_y = int((y1 + y2) / 2)

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2) 
    line_start = (x1, center_y)
    line_end = (x2, center_y)
    cv2.line(frame, line_start, line_end, (0, 255, 0), 2)  

input_image_path = r"D:\Samsthidhaa\Semester_7\Project_Phase_1\groundtruth_eyegaze\test\closed\images (4).jpeg"
frame = cv2.imread(input_image_path)

if frame is None:
    print("Error loading image.")
else:
    results = yolo_model(frame)

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            
            roi = frame[y1:y2, x1:x2]

            roi_tensor = preprocess_image(roi)

            gaze_class = predict_gaze(gaze_model, roi_tensor)
            gaze_label = class_mapping[gaze_class]

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)  # Green box for detection
            cv2.putText(frame, gaze_label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36, 255, 12), 2)

    cv2.imshow("Gaze Direction Prediction", frame)

    output_image_path = r"D:/Samsthidhaa/Semester_7/Project_Phase_1/conference_output/Closed.jpeg"
    cv2.imwrite(output_image_path, frame)

    cv2.waitKey(0)
    cv2.destroyAllWindows()



0: 448x640 1 eye, 41.9ms
Speed: 3.0ms preprocess, 41.9ms inference, 3.0ms postprocess per image at shape (1, 3, 448, 640)


In [ ]:
import cv2
from ultralytics import YOLO
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
import torch.nn as nn
import pyttsx3 

model = YOLO(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\bestEyeyolov8.pt")

yolo_model = YOLO(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\Eye_Alone_Detect.pt")

cap = cv2.VideoCapture(0)  
assert cap.isOpened(), "Error reading video file"

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

video_writer = cv2.VideoWriter("blink_count_output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

blink_count = 0
eye_was_closed = False
closed_time = 0  # Time counter for how long eyes have been closed
blinks_detected = 0  # Counter for detected blinks

communication_mode = False
communication_blink_count = 0  # Reset blink counter for communication mode

class InceptionResNetV2(nn.Module):
    def __init__(self, num_classes=1001):
        super(InceptionResNetV2, self).__init__()
        self.conv2d_1a = BasicConv2d(3, 32, kernel_size=3, stride=2)
        self.conv2d_2a = BasicConv2d(32, 32, kernel_size=3, stride=1)
        self.conv2d_2b = BasicConv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.maxpool_3a = nn.MaxPool2d(3, stride=2)
        self.conv2d_3b = BasicConv2d(64, 80, kernel_size=1, stride=1)
        self.conv2d_4a = BasicConv2d(80, 192, kernel_size=3, stride=1)
        self.maxpool_5a = nn.MaxPool2d(3, stride=2)
        self.mixed_5b = Mixed_5b()
        self.repeat = nn.Sequential(
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17)
        )
        self.mixed_6a = Mixed_6a()
        self.repeat_1 = nn.Sequential(
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10)
        )
        self.mixed_7a = Mixed_7a()
        self.repeat_2 = nn.Sequential(
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20)
        )
        self.block8 = Block8(noReLU=True)
        self.conv2d_7b = BasicConv2d(2080, 1536, kernel_size=1, stride=1)
        self.avgpool_1a = nn.AvgPool2d(8, count_include_pad=False)
        self.last_linear = nn.Linear(1536, num_classes)

    def features(self, input):
        x = self.conv2d_1a(input)
        x = self.conv2d_2a(x)
        x = self.conv2d_2b(x)
        x = self.maxpool_3a(x)
        x = self.conv2d_3b(x)
        x = self.conv2d_4a(x)
        x = self.maxpool_5a(x)
        x = self.mixed_5b(x)
        x = self.repeat(x)
        x = self.mixed_6a(x)
        x = self.repeat_1(x)
        x = self.mixed_7a(x)
        x = self.repeat_2(x)
        x = self.block8(x)
        x = self.conv2d_7b(x)
        return x

    def logits(self, features):
        x = self.avgpool_1a(features)
        x = x.view(x.size(0), -1)
        x = self.last_linear(x)
        return x

    def forward(self, input):
        x = self.features(input)
        x = self.logits(x)
        return x

def load_model(model_path, num_classes=4):
    model = InceptionResNetV2(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model

gaze_model = load_model(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\dataset\eye_gaze_model_2.pth")

def preprocess_image(roi):
    transform = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])
    image = Image.fromarray(roi)
    image = transform(image).unsqueeze(0) 
    return image

def predict_gaze(model, image_tensor):
    with torch.no_grad():
        output = model(image_tensor)
        _, predicted = torch.max(output, 1)
        return predicted.item()

class_mapping = {0: 'straight', 1: 'left', 2: 'right', 3: 'closed'}

def draw_eye_bounding_box_and_line(frame, bbox):
    x1, y1, x2, y2 = map(int, bbox)  
    center_y = int((y1 + y2) / 2) 

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  
    
    line_start = (x1, center_y)
    line_end = (x2, center_y)
    cv2.line(frame, line_start, line_end, (0, 255, 0), 2)  

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("Video frame is empty or video processing is done.")
        break
    
    results = model(frame)
    
    eye_detected = False

    for r in results[0].boxes:
        bbox = r.xyxy[0].tolist()  
        class_id = int(r.cls[0])
        eye_label = results[0].names[class_id]  # 'Closed Eyes' or 'Opened Eyes'
        
        draw_eye_bounding_box_and_line(frame, bbox)
        
        if eye_label == "closed eyes":
            eye_detected = True
            if not eye_was_closed:
                eye_was_closed = True  
                closed_time = 0  
            else:
                closed_time += 1 / fps 
            
            if closed_time >= 4:  # Check if eyes have been closed for 4 seconds
                print("Entered blink detection")
                cv2.putText(frame, "Communication Mode is activated!!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
                blink_count = 0  
                communication_mode = True 
                cv2.putText(frame, "Blink 4 times again to select your need!", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
                
                engine = pyttsx3.init()
                engine.say("Communication Mode is activated! Blink 4 times to select your need!")
                engine.runAndWait()


        elif eye_label == "opened eyes":
            eye_detected = True
            if eye_was_closed:
                blink_count += 1  
                blinks_detected += 1  
                print(f"Blinked! Count: {blink_count}")  
                eye_was_closed = False 

    if not eye_detected:
        eye_was_closed = False

    if communication_mode:
        if blink_count >= 4:  
            print("You selected water!")
            cv2.putText(frame, "You selected water! Do you want to proceed?", (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
            communication_blink_count = 0 
            
            engine.say("You selected water! Do you want to proceed?")
            engine.runAndWait()

            continue

        if blink_count >= 2:  
            print("Water need sent!")
            cv2.putText(frame, "Water need sent!", (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
            
            engine.say("I need water. Please provide me some")
            engine.runAndWait()

            communication_mode = False
            blink_count = 0  
            continue

    if blinks_detected >= 4:
        print("Mobility mode!")
            
        text_1 = "The Mobility Mode is Activated!!"
        text_2 = "Look Left to Move Left"
        text_3 = "Look Right to Move Right"
        text_4 = "Close Eyes to Take Reverse"

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            cv2.putText(frame, text_1, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, text_2, (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, text_3, (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, text_4, (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

            cv2.imshow('Blink Detection', frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    video_writer.write(frame)

    cv2.imshow('Blink Detection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()



KeyboardInterrupt



In [ ]:
import cv2
from ultralytics import YOLO
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
import torch.nn as nn
import pyttsx3  

model = YOLO(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\bestEyeyolov8.pt")

yolo_model = YOLO(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\Eye_Alone_Detect.pt")

cap = cv2.VideoCapture(0)
assert cap.isOpened(), "Error reading video file"

w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

video_writer = cv2.VideoWriter("blink_count_output.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

blink_count = 0
eye_was_closed = False
closed_time = 0  
blinks_detected = 0

communication_mode = False

class InceptionResNetV2(nn.Module):
    def __init__(self, num_classes=1001):
        super(InceptionResNetV2, self).__init__()
        self.conv2d_1a = BasicConv2d(3, 32, kernel_size=3, stride=2)
        self.conv2d_2a = BasicConv2d(32, 32, kernel_size=3, stride=1)
        self.conv2d_2b = BasicConv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.maxpool_3a = nn.MaxPool2d(3, stride=2)
        self.conv2d_3b = BasicConv2d(64, 80, kernel_size=1, stride=1)
        self.conv2d_4a = BasicConv2d(80, 192, kernel_size=3, stride=1)
        self.maxpool_5a = nn.MaxPool2d(3, stride=2)
        self.mixed_5b = Mixed_5b()
        self.repeat = nn.Sequential(
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17),
            Block35(scale=0.17)
        )
        self.mixed_6a = Mixed_6a()
        self.repeat_1 = nn.Sequential(
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10),
            Block17(scale=0.10)
        )
        self.mixed_7a = Mixed_7a()
        self.repeat_2 = nn.Sequential(
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20),
            Block8(scale=0.20)
        )
        self.block8 = Block8(noReLU=True)
        self.conv2d_7b = BasicConv2d(2080, 1536, kernel_size=1, stride=1)
        self.avgpool_1a = nn.AvgPool2d(8, count_include_pad=False)
        self.last_linear = nn.Linear(1536, num_classes)

    def features(self, input):
        x = self.conv2d_1a(input)
        x = self.conv2d_2a(x)
        x = self.conv2d_2b(x)
        x = self.maxpool_3a(x)
        x = self.conv2d_3b(x)
        x = self.conv2d_4a(x)
        x = self.maxpool_5a(x)
        x = self.mixed_5b(x)
        x = self.repeat(x)
        x = self.mixed_6a(x)
        x = self.repeat_1(x)
        x = self.mixed_7a(x)
        x = self.repeat_2(x)
        x = self.block8(x)
        x = self.conv2d_7b(x)
        return x

    def logits(self, features):
        x = self.avgpool_1a(features)
        x = x.view(x.size(0), -1)
        x = self.last_linear(x)
        return x

    def forward(self, input):
        x = self.features(input)
        x = self.logits(x)
        return x

def load_model(model_path, num_classes=4):
    model = InceptionResNetV2(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model

gaze_model = load_model(r"D:\Samsthidhaa\Semester_7\Project_Phase_1\dataset\eye_gaze_model_2.pth")

def preprocess_image(roi):
    transform = transforms.Compose([
        transforms.Resize((299, 299)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])
    image = Image.fromarray(roi)
    image = transform(image).unsqueeze(0)  # Add batch dimension
    return image

def predict_gaze(model, image_tensor):
    with torch.no_grad():
        output = model(image_tensor)
        _, predicted = torch.max(output, 1)
        return predicted.item()

class_mapping = {0: 'straight', 1: 'left', 2: 'right', 3: 'closed'}

def draw_eye_bounding_box_and_line(frame, bbox):
    x1, y1, x2, y2 = map(int, bbox) 
    center_y = int((y1 + y2) / 2) 

    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)  
    
    line_start = (x1, center_y)
    line_end = (x2, center_y)
    cv2.line(frame, line_start, line_end, (0, 255, 0), 2)

engine = pyttsx3.init()

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        print("Video frame is empty or video processing is done.")
        break
    
    results = model(frame)
    
    eye_detected = False

    for r in results[0].boxes:
        bbox = r.xyxy[0].tolist() 
        class_id = int(r.cls[0])
        eye_label = results[0].names[class_id]  # 'Closed Eyes' or 'Opened Eyes'
        
        draw_eye_bounding_box_and_line(frame, bbox)
        
        if eye_label == "closed eyes":
            eye_detected = True
            if not eye_was_closed:
                eye_was_closed = True 
                closed_time = 0 
            else:
                closed_time += 1 / fps
            
            if closed_time >= 4:  # Check if eyes have been closed for 4 seconds
                print("Entered blink detection")
                cv2.putText(frame, "Communication Mode is activated!!", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
                
                communication_mode = True
                blink_count = 0  
                cv2.putText(frame, "Blink 4 times again to select your need!", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
                
                if blink_count >= 4:  # Blink 4 times to select water
                    print("You selected water!")
                    cv2.putText(frame, "You selected water! Do you want to proceed?", (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
                    blink_count = 0 

                if blink_count >= 2: 
                    print("Water need sent!")
                    cv2.putText(frame, "Water need sent!", (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

                    engine.say("I need water. Please provide me some.")
                    engine.runAndWait()
                    
                    break  

        elif eye_label == "opened eyes":
            eye_detected = True
            if eye_was_closed:
                blink_count += 1  
                print(f"Blinked! Count: {blink_count}")  
                eye_was_closed = False

    if not eye_detected:
        eye_was_closed = False

    if blinks_detected >= 4:
        print("Mobility mode!")
        

    video_writer.write(frame)

    cv2.imshow('Blink Detection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()



0: 480x640 2 opened eyess, 114.2ms
Speed: 4.0ms preprocess, 114.2ms inference, 151.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 46.9ms
Speed: 3.0ms preprocess, 46.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 45.9ms
Speed: 3.0ms preprocess, 45.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 46.4ms
Speed: 2.0ms preprocess, 46.4ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 41.4ms
Speed: 2.0ms preprocess, 41.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 39.4ms
Speed: 2.0ms preprocess, 39.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 40.9ms
Speed: 2.0ms preprocess, 40.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 39.9ms
Speed: 2.0ms preprocess, 39.9ms infer

Speed: 3.0ms preprocess, 38.2ms inference, 2.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 37.9ms
Speed: 2.0ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.4ms
Speed: 3.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.4ms
Speed: 2.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 closed eyess, 39.3ms
Speed: 2.0ms preprocess, 39.3ms inference, 3.0ms postprocess per image at sh


0: 480x640 2 opened eyess, 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 4.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 closed eyess, 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 closed eyess, 38.4ms
Speed: 3.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 closed eyess, 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.9ms
Speed: 2.0ms preprocess, 38.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)
Blinked! Count: 13

0: 480x640 2 opened eyess, 37.9ms
Speed: 2.5ms preprocess, 37.9ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 38.4ms
Speed: 2.0ms preprocess, 38.4ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 opened eyess, 37.9ms
Speed: 3.0ms preproces